# Enhance Louvain结果

In [ ]:
# 对误标注结果修正
# 周三DoS Hulk和Dos GoldenEye、DoS Slowhttptest修正
# start_time = datetime.strptime('2017-07-05 10:30:00', '%Y-%m-%d %H:%M:%S')
# end_time = datetime.strptime('2017-07-05 10:40:00', '%Y-%m-%d %H:%M:%S')
# conditional_Wednesday = (snort_df_filtered['Label'] == 'BENIGN') & (snort_df_filtered['real_time'] >= start_time) & (snort_df_filtered['real_time'] < end_time) & (snort_df_filtered['dst_addr']=='192.168.10.50') & (snort_df_filtered['dst_port']==80) & (snort_df_filtered['src_addr']=='172.16.0.1')
# snort_df_correct.loc[conditional_Wednesday, 'Label'] = 'DoS Slowhttptest'

# # 筛选10:40到11:10之间的数据
# start_time = datetime.strptime('2017-07-05 10:40:00', '%Y-%m-%d %H:%M:%S')
# end_time = datetime.strptime('2017-07-05 11:10:00', '%Y-%m-%d %H:%M:%S')
# conditional_Wednesday = (snort_df_filtered['Label'] == 'BENIGN') & (snort_df_filtered['real_time'] >= start_time) & (snort_df_filtered['real_time'] < end_time) & (snort_df_filtered['dst_addr']=='192.168.10.50') & (snort_df_filtered['dst_port']==80) & (snort_df_filtered['src_addr']=='172.16.0.1')
# snort_df_correct.loc[conditional_Wednesday, 'Label'] = 'DoS Hulk'

# start_time = datetime.strptime('2017-07-05 11:10:00', '%Y-%m-%d %H:%M:%S')
# end_time = datetime.strptime('2017-07-05 11:20:00', '%Y-%m-%d %H:%M:%S')
# conditional_Wednesday = (snort_df_filtered['Label'] == 'BENIGN') & (snort_df_filtered['real_time'] >= start_time) & (snort_df_filtered['real_time'] <= end_time) & (snort_df_filtered['dst_addr']=='192.168.10.50') & (snort_df_filtered['dst_port']==80) & (snort_df_filtered['src_addr']=='172.16.0.1')
# snort_df_correct.loc[conditional_Wednesday, 'Label'] = 'DoS GoldenEye'

In [ ]:
# 训练集
coo_train_path = f"../..//dataset/graph-format-data/graph_train.csv"
coo_train_graph_df = pd.read_csv(coo_train_path)
graph_all_data = coo_train_graph_df
# 统计每个簇的告警数量
edges_label = graph_all_data['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num = sum(one_edge_label.values())
    warn_num_list.append(warn_num)
graph_all_data['warn_num'] = warn_num_list

multiple_risk_graph = graph_all_data[graph_all_data['group_label'].apply(lambda x: len(eval(x)) >= 2)]
multiple_risk_graph = multiple_risk_graph.reset_index(drop=True)
multiple_risk_graph_percent = len(multiple_risk_graph) / len(graph_all_data)
multiple_risk_alert_percent = multiple_risk_graph['warn_num'].sum() / graph_all_data['warn_num'].sum()

print('---------------------------------RESULTS----------------------------------------------------')
print("图划分单独的风险/攻击类型簇占比: {:.2%}, 告警占比{:.2%}".format(1-multiple_risk_graph_percent, 1-multiple_risk_alert_percent))
# 按照warn_num对multiple_risk_graph进行排序
multiple_risk_graph = multiple_risk_graph.sort_values(by='warn_num', ascending=False)
# 根据edge_label拿到每个label的数量
edges_label = multiple_risk_graph['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num_dict = {}
    for key,value in one_edge_label.items():
        label_name = key[1]
        if label_name in warn_num_dict.keys():
            warn_num_dict[label_name] += value
        else:
            warn_num_dict[label_name] = value
            
    warn_num_list.append(str(warn_num_dict))

multiple_risk_graph['warn_num_dicts'] = warn_num_list
multiple_risk_graph.reset_index(drop=True)

In [21]:
coo_train_graph_df.loc[coo_train_graph_df['id']==4855]
coo_train_correct_df = coo_train_graph_df.copy()

In [23]:
# 进行修正
correct_id = [4855,4896,4994,4989]
# 筛选出来这些
graph_all_data_filter = graph_all_data[graph_all_data['id'].isin(correct_id)]
graph_all_data_filter.reset_index(drop=True, inplace=True)
# 修正edge_label&group_label
for i in range(len(graph_all_data_filter)):
    id = graph_all_data_filter.loc[i,'id']
    now_edge_label = eval(graph_all_data_filter.loc[i,'edge_label'])
    now_group_label = eval(graph_all_data_filter.loc[i,'group_label'])
    correct_edge_label = {}

    if i < 2:
        # 修正为'DoS Hulk'
        correct_group_label = {'DoS Hulk'}
        for key,value in now_edge_label.items():
            attack_label = key[1]
            if attack_label == 'BENIGN':
                if (key[0],'DoS Hulk') in correct_edge_label:
                    correct_edge_label[(key[0],'DoS Hulk')] += value
                else:
                    correct_edge_label[(key[0],'DoS Hulk')] = value
            else:
                if key in correct_edge_label:
                    correct_edge_label[key] += value
                else:
                    correct_edge_label[key] = value
    else:
        correct_group_label = {'DoS GoldenEye'}
        for key,value in now_edge_label.items():
            attack_label = key[1]
            if attack_label == 'BENIGN':
                if (key[0],'DoS GoldenEye') in correct_edge_label:
                    correct_edge_label[(key[0],'DoS GoldenEye')] += value
                else:
                    correct_edge_label[(key[0],'DoS GoldenEye')] = value
            else:
                if key in correct_edge_label:
                    correct_edge_label[key] += value
                else:
                    correct_edge_label[key] = value
    print(now_edge_label)
    print(correct_edge_label)
    #  修正tran_df
    coo_train_correct_df.loc[coo_train_correct_df['id'] == id,'group_label'] = str(correct_group_label)
    coo_train_correct_df.loc[coo_train_correct_df['id'] == id,'edge_label'] = str(correct_edge_label)

{(0, 'DoS Hulk'): 37007, (0, 'BENIGN'): 1, (1, 'DoS Hulk'): 28563, (2, 'DoS Hulk'): 10729, (3, 'DoS Hulk'): 18301, (4, 'DoS Hulk'): 19, (4, 'BENIGN'): 9, (5, 'DoS Hulk'): 3, (6, 'DoS Hulk'): 26216, (6, 'BENIGN'): 1, (7, 'DoS Hulk'): 2068, (8, 'DoS Hulk'): 80, (9, 'DoS Hulk'): 1, (10, 'DoS Hulk'): 4}
{(0, 'DoS Hulk'): 37008, (1, 'DoS Hulk'): 28563, (2, 'DoS Hulk'): 10729, (3, 'DoS Hulk'): 18301, (4, 'DoS Hulk'): 28, (5, 'DoS Hulk'): 3, (6, 'DoS Hulk'): 26217, (7, 'DoS Hulk'): 2068, (8, 'DoS Hulk'): 80, (9, 'DoS Hulk'): 1, (10, 'DoS Hulk'): 4}
{(0, 'DoS Hulk'): 28717, (0, 'BENIGN'): 2, (1, 'DoS Hulk'): 23015, (2, 'BENIGN'): 23, (2, 'DoS Hulk'): 4, (3, 'DoS Hulk'): 1, (4, 'DoS Hulk'): 14446, (4, 'BENIGN'): 1, (5, 'DoS Hulk'): 19216, (5, 'BENIGN'): 1, (6, 'DoS Hulk'): 1402, (7, 'DoS Hulk'): 441, (8, 'BENIGN'): 4}
{(0, 'DoS Hulk'): 28719, (1, 'DoS Hulk'): 23015, (2, 'DoS Hulk'): 27, (3, 'DoS Hulk'): 1, (4, 'DoS Hulk'): 14447, (5, 'DoS Hulk'): 19217, (6, 'DoS Hulk'): 1402, (7, 'DoS Hulk'): 4

In [30]:
coo_train_correct_df.to_csv("../..//dataset/graph-format-data/noise_label_data/train_correct_df.csv",index=False)

In [ ]:
# 类似的对测试集做修正

In [27]:
import pandas as pd
import numpy as np
import ast

# 训练集
coo_train_path = f"../..//dataset/graph-format-data/noise_label_data/train_correct_df.csv"
coo_train_graph_df = pd.read_csv(coo_train_path)
# 测试集
coo_test_path = f"../..//dataset/graph-format-data/graph_test.csv"
coo_test_graph_df = pd.read_csv(coo_test_path)
# 按行拼接（垂直堆叠）
coo_graph_df = pd.concat([coo_train_graph_df, coo_test_graph_df], axis=0).reset_index(drop=True) # coo数据拼接

# 重新生成唯一id（覆盖原有id列）
coo_graph_df["id"] = range(len(coo_graph_df))

graph_all_data = coo_graph_df
# 统计每个簇的告警数量
edges_label = graph_all_data['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num = sum(one_edge_label.values())
    warn_num_list.append(warn_num)
graph_all_data['warn_num'] = warn_num_list

# # 去除 'BENIGN'，如果去除后为空，则保留原来的函数：将字符串转换为集合，并去掉 'BENIGN'（如果剩下为空则保留原集合）
# import ast
# def parse_and_strip_benign(label_str):
#     label_set = ast.literal_eval(label_str)  # 安全地解析字符串为集合
#     new_set = label_set - {'BENIGN'}

#     return str(new_set) if new_set else str(label_set)

# # 应用函数处理每一项
# graph_all_data['cleaned_group_label'] = graph_all_data['group_label'].apply(parse_and_strip_benign)
graph_all_data = graph_all_data.reset_index(drop=True)
print("CIC-IDS2017数据集")
print("安全事件的数量为：", len(graph_all_data))

CIC-IDS2017数据集
安全事件的数量为： 10605


In [28]:
graph_all_data['warn_num'].sum()

685561

In [29]:
multiple_risk_graph = graph_all_data[graph_all_data['group_label'].apply(lambda x: len(eval(x)) >= 2)]
multiple_risk_graph = multiple_risk_graph.reset_index(drop=True)
multiple_risk_graph_percent = len(multiple_risk_graph) / len(graph_all_data)
multiple_risk_alert_percent = multiple_risk_graph['warn_num'].sum() / graph_all_data['warn_num'].sum()

print('---------------------------------RESULTS----------------------------------------------------')
print("图划分单独的风险/攻击类型簇占比: {:.2%}, 告警占比{:.2%}".format(1-multiple_risk_graph_percent, 1-multiple_risk_alert_percent))

---------------------------------RESULTS----------------------------------------------------
图划分单独的风险/攻击类型簇占比: 99.40%, 告警占比91.78%


In [ ]:
# 按照warn_num对multiple_risk_graph进行排序
multiple_risk_graph = multiple_risk_graph.sort_values(by='warn_num', ascending=False)
# 根据edge_label拿到每个label的数量
edges_label = multiple_risk_graph['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num_dict = {}
    for key,value in one_edge_label.items():
        label_name = key[1]
        if label_name in warn_num_dict.keys():
            warn_num_dict[label_name] += value
        else:
            warn_num_dict[label_name] = value
            
    warn_num_list.append(str(warn_num_dict))

multiple_risk_graph['warn_num_dicts'] = warn_num_list
multiple_risk_graph.reset_index(drop=True)

In [53]:
import pandas as pd
import numpy as np
import networkx as nx
import ast

def compute_graph_stats(row):
    # 将字符串形式的 edge_index 转为列表
    edges = ast.literal_eval(row['edge_index'])  # COO格式：[sources, targets]

    # 构建 NetworkX 的有向图
    G = nx.DiGraph()
    G.add_edges_from(edges)

    # 计算平均度（入度+出度除以节点数）
    degrees = [deg for _, deg in G.degree()]
    avg_degree = np.mean(degrees)

    # 最大强连通分量大小
    scc = list(nx.strongly_connected_components(G))
    max_scc_size = max([len(c) for c in scc]) if scc else 0

    # 最大弱连通分量大小
    wcc = list(nx.weakly_connected_components(G))
    max_wcc_size = max([len(c) for c in wcc]) if wcc else 0

    return pd.Series({
        'avg_degree': avg_degree,
        'max_scc_size': max_scc_size,
        'max_wcc_size': max_wcc_size
    })
# 示例：对整个 DataFrame 应用
graph_all_data[['avg_degree', 'max_scc_size', 'max_wcc_size']] = graph_all_data.apply(compute_graph_stats, axis=1)

In [54]:
global_avg_degree = graph_all_data['avg_degree'].mean()
global_max_scc_size = graph_all_data['max_scc_size'].mean()
global_max_wcc_size = graph_all_data['max_wcc_size'].mean()

# 打印结果
print(f"全局平均度：{global_avg_degree:.2f}")
print(f"最大强连通分量大小（平均）：{global_max_scc_size:.2f}")
print(f"最大弱连通分量大小（平均）：{global_max_wcc_size:.2f}")

全局平均度：1.37
最大强连通分量大小（平均）：1.12
最大弱连通分量大小（平均）：8.13


# Louvain结果

In [28]:
import pandas as pd
import numpy as np
import ast
# 训练集
coo_train_graph_df = pd.read_csv("../..//dataset/CIC-IDS2017-format-data/correct_data/Louvain/graph_train.csv")
# 测试集
coo_test_graph_df = pd.read_csv("../..//dataset/CIC-IDS2017-format-data/correct_data/Louvain/graph_test.csv")
# 按行拼接（垂直堆叠）
coo_graph_df = pd.concat([coo_train_graph_df, coo_test_graph_df], axis=0).reset_index(drop=True) # coo数据拼接

# 重新生成唯一id（覆盖原有id列）
coo_graph_df["id"] = range(len(coo_graph_df))

# 联合起来
graph_all_data_louvain = coo_graph_df
# 统计每个簇的告警数量
edges_label = graph_all_data_louvain['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num = sum(one_edge_label.values())
    warn_num_list.append(warn_num)
graph_all_data_louvain['warn_num'] = warn_num_list


In [29]:
graph_all_data_louvain['warn_num'].sum()

659825

In [30]:
len(graph_all_data_louvain)

1773

In [31]:
multiple_risk_graph = graph_all_data_louvain[graph_all_data_louvain['group_label'].apply(lambda x: len(eval(x)) >= 2)]
multiple_risk_graph = multiple_risk_graph.reset_index(drop=True)
multiple_risk_graph_percent = len(multiple_risk_graph) / len(graph_all_data_louvain)
multiple_risk_alert_percent = multiple_risk_graph['warn_num'].sum() / graph_all_data_louvain['warn_num'].sum()

print('---------------------------------RESULTS----------------------------------------------------')
print("图划分单独的风险/攻击类型簇占比: {:.2%}, 告警占比{:.2%}".format(1-multiple_risk_graph_percent, 1-multiple_risk_alert_percent))

---------------------------------RESULTS----------------------------------------------------
图划分单独的风险/攻击类型簇占比: 98.03%, 告警占比54.17%


In [ ]:
# 按照warn_num对multiple_risk_graph进行排序
multiple_risk_graph = multiple_risk_graph.sort_values(by='warn_num', ascending=False)
# 根据edge_label拿到每个label的数量
edges_label = multiple_risk_graph['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num_dict = {}
    for key,value in one_edge_label.items():
        label_name = key[1]
        if label_name in warn_num_dict.keys():
            warn_num_dict[label_name] += value
        else:
            warn_num_dict[label_name] = value
            
    warn_num_list.append(str(warn_num_dict))

multiple_risk_graph['warn_num_dicts'] = warn_num_list
multiple_risk_graph.reset_index(drop=True)
# id	group_num	edge_index	node_attr	edge_attr	group_attr	edge_label	group_label	nodes	start_time	end_time	warn_num	warn_num_dicts
multiple_risk_graph = multiple_risk_graph[['id','edge_index','group_num','node_attr','group_attr','edge_label','group_label','nodes','start_time','end_time','warn_num','warn_num_dicts']]

multiple_risk_graph.to_csv('multiple_attack_louvain_graph.csv', index=False)

# Girvan_Newman

In [ ]:
import pandas as pd
import numpy as np
import ast
# 训练集
coo_train_graph_df = pd.read_csv("../..//dataset/CIC-IDS2017-format-data/correct_data/GN/graph_train.csv")
# 测试集
coo_test_graph_df = pd.read_csv("../..//dataset/CIC-IDS2017-format-data/correct_data/GN/graph_test.csv")
# 按行拼接（垂直堆叠）
coo_graph_df = pd.concat([coo_train_graph_df, coo_test_graph_df], axis=0).reset_index(drop=True) # coo数据拼接

# 重新生成唯一id（覆盖原有id列）
coo_graph_df["id"] = range(len(coo_graph_df))

# 联合起来
graph_all_data_Girvan_Newman = coo_graph_df
# 统计每个簇的告警数量
edges_label = graph_all_data_Girvan_Newman['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num = sum(one_edge_label.values())
    warn_num_list.append(warn_num)
graph_all_data_Girvan_Newman['warn_num'] = warn_num_list

# 去除unlabel影响,一个簇包含unlabel和另一个风险等级则认为该簇为这个风险等级，其他多风险等级的情况无法处理
for index_count in range(len(graph_all_data_Girvan_Newman)):
    group_label = graph_all_data_Girvan_Newman.loc[index_count,'group_label']
    group_label_set = eval(group_label)
    if 'unlabel' in group_label_set:
        group_label_set.remove('unlabel')
    if len(group_label_set) == 0:
        group_label_set = "{'unlabel'}"
    graph_all_data_Girvan_Newman.loc[index_count,'group_label'] = str(group_label_set)

############################删除无标签数据##########################
graph_all_data_Girvan_Newman = graph_all_data_Girvan_Newman[graph_all_data_Girvan_Newman['group_label'] != "{'unlabel'}"]
graph_all_data_Girvan_Newman = graph_all_data_Girvan_Newman.reset_index(drop=True)
graph_all_data_Girvan_Newman

In [35]:
graph_all_data_Girvan_Newman['warn_num'].sum()

603039

In [36]:
multiple_risk_graph = graph_all_data_Girvan_Newman[graph_all_data_Girvan_Newman['group_label'].apply(lambda x: len(eval(x)) >= 2)]
multiple_risk_graph = multiple_risk_graph.reset_index(drop=True)
multiple_risk_graph_percent = len(multiple_risk_graph) / len(graph_all_data_Girvan_Newman)
multiple_risk_alert_percent = multiple_risk_graph['warn_num'].sum() / graph_all_data_Girvan_Newman['warn_num'].sum()

print('---------------------------------RESULTS----------------------------------------------------')
print("图划分单独的风险/攻击类型簇占比: {:.2%}, 告警占比{:.2%}".format(1-multiple_risk_graph_percent, 1-multiple_risk_alert_percent))

---------------------------------RESULTS----------------------------------------------------
图划分单独的风险/攻击类型簇占比: 98.22%, 告警占比49.72%


In [38]:
# 按照warn_num对multiple_risk_graph进行排序
multiple_risk_graph = multiple_risk_graph.sort_values(by='warn_num', ascending=False)
# 根据edge_label拿到每个label的数量
edges_label = multiple_risk_graph['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num_dict = {}
    for key,value in one_edge_label.items():
        label_name = key[1]
        if label_name in warn_num_dict.keys():
            warn_num_dict[label_name] += value
        else:
            warn_num_dict[label_name] = value
            
    warn_num_list.append(str(warn_num_dict))

multiple_risk_graph['warn_num_dicts'] = warn_num_list
multiple_risk_graph.reset_index(drop=True)
# id	group_num	edge_index	node_attr	edge_attr	group_attr	edge_label	group_label	nodes	start_time	end_time	warn_num	warn_num_dicts
multiple_risk_graph = multiple_risk_graph[['id','edge_index','group_num','node_attr','group_attr','edge_label','group_label','nodes','start_time','end_time','warn_num','warn_num_dicts']]

multiple_risk_graph.to_csv('multiple_attack_GN_graph.csv', index=False)

# label_propagation

In [ ]:
import pandas as pd
import numpy as np
import ast
# 训练集
coo_train_graph_df = pd.read_csv("../../dataset/CIC-IDS2017-format-data/correct_data/LPA/graph_train.csv")
# 测试集
coo_test_graph_df = pd.read_csv("../../dataset/CIC-IDS2017-format-data/correct_data/LPA/graph_test.csv")
# 按行拼接（垂直堆叠）
coo_graph_df = pd.concat([coo_train_graph_df, coo_test_graph_df], axis=0).reset_index(drop=True) # coo数据拼接

# 重新生成唯一id（覆盖原有id列）
coo_graph_df["id"] = range(len(coo_graph_df))

# 联合起来
graph_all_data_label_propagation = coo_graph_df
# 统计每个簇的告警数量
edges_label = graph_all_data_label_propagation['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num = sum(one_edge_label.values())
    warn_num_list.append(warn_num)
graph_all_data_label_propagation['warn_num'] = warn_num_list

# 去除unlabel影响,一个簇包含unlabel和另一个风险等级则认为该簇为这个风险等级，其他多风险等级的情况无法处理
for index_count in range(len(graph_all_data_label_propagation)):
    group_label = graph_all_data_label_propagation.loc[index_count,'group_label']
    group_label_set = eval(group_label)
    if 'unlabel' in group_label_set:
        group_label_set.remove('unlabel')
    if len(group_label_set) == 0:
        group_label_set = "{'unlabel'}"
    graph_all_data_label_propagation.loc[index_count,'group_label'] = str(group_label_set)

############################删除无标签数据##########################
graph_all_data_label_propagation = graph_all_data_label_propagation[graph_all_data_label_propagation['group_label'] != "{'unlabel'}"]
graph_all_data_label_propagation = graph_all_data_label_propagation.reset_index(drop=True)
graph_all_data_label_propagation

In [2]:
graph_all_data_label_propagation['warn_num'].sum()

675044

In [3]:
multiple_risk_graph = graph_all_data_label_propagation[graph_all_data_label_propagation['group_label'].apply(lambda x: len(eval(x)) >= 2)]
multiple_risk_graph = multiple_risk_graph.reset_index(drop=True)
multiple_risk_graph_percent = len(multiple_risk_graph) / len(graph_all_data_label_propagation)
multiple_risk_alert_percent = multiple_risk_graph['warn_num'].sum() / graph_all_data_label_propagation['warn_num'].sum()

print('---------------------------------RESULTS----------------------------------------------------')
print("图划分单独的风险/攻击类型簇占比: {:.2%}, 告警占比{:.2%}".format(1-multiple_risk_graph_percent, 1-multiple_risk_alert_percent))

---------------------------------RESULTS----------------------------------------------------
图划分单独的风险/攻击类型簇占比: 97.75%, 告警占比55.04%


In [4]:
# 按照warn_num对multiple_risk_graph进行排序
multiple_risk_graph = multiple_risk_graph.sort_values(by='warn_num', ascending=False)
# 根据edge_label拿到每个label的数量
edges_label = multiple_risk_graph['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num_dict = {}
    for key,value in one_edge_label.items():
        label_name = key[1]
        if label_name in warn_num_dict.keys():
            warn_num_dict[label_name] += value
        else:
            warn_num_dict[label_name] = value
            
    warn_num_list.append(str(warn_num_dict))

multiple_risk_graph['warn_num_dicts'] = warn_num_list
multiple_risk_graph.reset_index(drop=True)
# id	group_num	edge_index	node_attr	edge_attr	group_attr	edge_label	group_label	nodes	start_time	end_time	warn_num	warn_num_dicts
multiple_risk_graph = multiple_risk_graph[['id','edge_index','group_num','node_attr','group_attr','edge_label','group_label','nodes','start_time','end_time','warn_num','warn_num_dicts']]

multiple_risk_graph.to_csv('../../src/evluation/question_log/multiple_attack_LPA_graph.csv', index=False)

# 双次Louvain

In [ ]:
import pandas as pd
import numpy as np
import ast
# 训练集
coo_train_graph_df = pd.read_csv("../../dataset/CIC-IDS2017-format-data/correct_data/Second_Louvain/graph_train.csv")
# 测试集
coo_test_graph_df = pd.read_csv("../../dataset/CIC-IDS2017-format-data/correct_data/Second_Louvain/graph_test.csv")
# 按行拼接（垂直堆叠）
coo_graph_df = pd.concat([coo_train_graph_df, coo_test_graph_df], axis=0).reset_index(drop=True) # coo数据拼接

# 重新生成唯一id（覆盖原有id列）
coo_graph_df["id"] = range(len(coo_graph_df))

# 联合起来
graph_all_data_louvain = coo_graph_df
# 统计每个簇的告警数量
edges_label = graph_all_data_louvain['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num = sum(one_edge_label.values())
    warn_num_list.append(warn_num)
graph_all_data_louvain['warn_num'] = warn_num_list

# 去除unlabel影响,一个簇包含unlabel和另一个风险等级则认为该簇为这个风险等级，其他多风险等级的情况无法处理
for index_count in range(len(graph_all_data_louvain)):
    group_label = graph_all_data_louvain.loc[index_count,'group_label']
    group_label_set = eval(group_label)
    if 'unlabel' in group_label_set:
        group_label_set.remove('unlabel')
    if len(group_label_set) == 0:
        group_label_set = "{'unlabel'}"
    graph_all_data_louvain.loc[index_count,'group_label'] = str(group_label_set)

############################删除无标签数据##########################
graph_all_data_louvain = graph_all_data_louvain[graph_all_data_louvain['group_label'] != "{'unlabel'}"]
graph_all_data_louvain = graph_all_data_louvain.reset_index(drop=True)
graph_all_data_louvain

In [6]:
graph_all_data_louvain['warn_num'].sum()

645102

In [9]:
multiple_risk_graph = graph_all_data_louvain[graph_all_data_louvain['group_label'].apply(lambda x: len(eval(x)) >= 2)]
multiple_risk_graph = multiple_risk_graph.reset_index(drop=True)
multiple_risk_graph_percent = len(multiple_risk_graph) / len(graph_all_data_louvain)
multiple_risk_alert_percent = multiple_risk_graph['warn_num'].sum() / graph_all_data_louvain['warn_num'].sum()

print('---------------------------------RESULTS----------------------------------------------------')
print("图划分单独的风险/攻击类型簇占比: {:.2%}, 告警占比{:.2%}".format(1-multiple_risk_graph_percent, 1-multiple_risk_alert_percent))

---------------------------------RESULTS----------------------------------------------------
图划分单独的风险/攻击类型簇占比: 98.55%, 告警占比53.20%


In [10]:
# 按照warn_num对multiple_risk_graph进行排序
multiple_risk_graph = multiple_risk_graph.sort_values(by='warn_num', ascending=False)
# 根据edge_label拿到每个label的数量
edges_label = multiple_risk_graph['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num_dict = {}
    for key,value in one_edge_label.items():
        label_name = key[1]
        if label_name in warn_num_dict.keys():
            warn_num_dict[label_name] += value
        else:
            warn_num_dict[label_name] = value
            
    warn_num_list.append(str(warn_num_dict))

multiple_risk_graph['warn_num_dicts'] = warn_num_list
multiple_risk_graph.reset_index(drop=True)
# id	group_num	edge_index	node_attr	edge_attr	group_attr	edge_label	group_label	nodes	start_time	end_time	warn_num	warn_num_dicts
multiple_risk_graph = multiple_risk_graph[['id','edge_index','group_num','node_attr','group_attr','edge_label','group_label','nodes','start_time','end_time','warn_num','warn_num_dicts']]

multiple_risk_graph.to_csv('../../src/evluation/question_log/multiple_attack_DoubleLouvain_graph.csv', index=False)

# Leiden

In [50]:
import pandas as pd
import numpy as np
import ast
# 训练集
coo_train_graph_df = pd.read_csv("../..//dataset/CIC-IDS2017-format-data/correct_data/Leiden/graph_train.csv")
# 测试集
coo_test_graph_df = pd.read_csv("../..//dataset/CIC-IDS2017-format-data/correct_data/Leiden/graph_test.csv")
# 按行拼接（垂直堆叠）
coo_graph_df = pd.concat([coo_train_graph_df, coo_test_graph_df], axis=0).reset_index(drop=True) # coo数据拼接

# 重新生成唯一id（覆盖原有id列）
coo_graph_df["id"] = range(len(coo_graph_df))

# 联合起来
graph_all_data_louvain = coo_graph_df
# 统计每个簇的告警数量
edges_label = graph_all_data_louvain['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num = sum(one_edge_label.values())
    warn_num_list.append(warn_num)
graph_all_data_louvain['warn_num'] = warn_num_list

# 去除unlabel影响,一个簇包含unlabel和另一个风险等级则认为该簇为这个风险等级，其他多风险等级的情况无法处理
for index_count in range(len(graph_all_data_louvain)):
    group_label = graph_all_data_louvain.loc[index_count,'group_label']
    group_label_set = eval(group_label)
    if 'unlabel' in group_label_set:
        group_label_set.remove('unlabel')
    if len(group_label_set) == 0:
        group_label_set = "{'unlabel'}"
    graph_all_data_louvain.loc[index_count,'group_label'] = str(group_label_set)

############################删除无标签数据##########################
graph_all_data_louvain = graph_all_data_louvain[graph_all_data_louvain['group_label'] != "{'unlabel'}"]
graph_all_data_louvain = graph_all_data_louvain.reset_index(drop=True)
graph_all_data_louvain

,id,group_num,edge_index,node_attr,edge_attr,group_attr,edge_label,group_label,nodes,start_time,end_time,warn_num
0,0,0,"[[0, 1], [0, 2], [0, 3], [4, 1], [4, 3], [4, 3]]","[[1, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 1, 0, 0, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[3, 0.3333333333333333, 0.0, 0.0, 0.6666666666...","{(0, 'BENIGN'): 4, (1, 'BENIGN'): 1, (2, 'BENI...",{'BENIGN'},"[{'id': 0, 'ip': '192.168.10.9'}, {'id': 1, 'i...",2017-07-03 08:50:00,2017-07-03 09:00:00,63
1,1,1,"[[0, 1], [1, 0], [2, 3], [1, 2], [1, 2], [1, 2...","[[0, 1, 0, 0, 0, 0, 0, 0, 0, 1], [1, 0, 0, 1, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[3, 0.16666666666666666, 0.3333333333333333, 0...","{(0, 'BENIGN'): 3, (1, 'BENIGN'): 1, (2, 'BENI...",{'BENIGN'},"[{'id': 0, 'ip': '192.168.10.1'}, {'id': 1, 'i...",2017-07-03 08:50:00,2017-07-03 09:00:00,108
2,2,2,"[[0, 1], [2, 1]]","[[0, 1, 0, 0, 0, 0, 0, 0, 0, 1], [1, 0, 0, 0, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0....","{(0, 'BENIGN'): 1, (1, 'BENIGN'): 1}",{'BENIGN'},"[{'id': 0, 'ip': 'fe80::8cc1:7756:5ffc:823f'},...",2017-07-03 08:50:00,2017-07-03 09:00:00,2
3,3,3,"[[0, 1], [0, 2]]","[[1, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 1, 0, 0, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0....","{(0, 'BENIGN'): 1, (1, 'BENIGN'): 2}",{'BENIGN'},"[{'id': 0, 'ip': '192.168.10.50'}, {'id': 1, '...",2017-07-03 08:50:00,2017-07-03 09:00:00,3
4,4,0,"[[0, 1], [2, 1], [3, 1], [3, 1], [3, 1], [1, 4...","[[0, 1, 0, 0, 0, 0, 0, 0, 0, 1], [1, 0, 0, 0, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0.3956043956043956, 0.3956043956043956, 0....","{(0, 'BENIGN'): 1, (1, 'BENIGN'): 1, (2, 'BENI...",{'BENIGN'},"[{'id': 0, 'ip': '23.50.75.27'}, {'id': 1, 'ip...",2017-07-03 09:00:00,2017-07-03 09:10:00,29
...,...,...,...,...,...,...,...,...,...,...,...,...
2012,2012,3,"[[0, 1], [2, 3], [3, 2], [4, 5], [1, 6], [1, 6...","[[0, 1, 0, 0, 0, 0, 0, 0, 0, 1], [1, 0, 0, 1, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[6, 0.0, 0.1, 0.25, 0.0, 0.0, 0.0, 0.25, 0.35,...","{(0, 'BENIGN'): 1, (1, 'BENIGN'): 1, (2, 'BENI...",{'BENIGN'},"[{'id': 0, 'ip': '192.168.10.1'}, {'id': 1, 'i...",2017-07-07 17:00:00,2017-07-07 17:10:00,17187
2013,2013,4,"[[0, 1], [1, 2], [1, 0], [1, 3], [1, 4], [5, 1...","[[0, 1, 0, 0, 0, 0, 0, 0, 0, 1], [1, 0, 0, 0, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0.21428571428571427, 0.42857142857142855, ...","{(0, 'BENIGN'): 1, (1, 'BENIGN'): 2, (2, 'BENI...",{'BENIGN'},"[{'id': 0, 'ip': '23.208.97.124'}, {'id': 1, '...",2017-07-07 17:00:00,2017-07-07 17:10:00,10
2014,2014,5,"[[0, 1], [2, 1], [3, 1], [4, 1], [5, 1], [6, 1]]","[[0, 1, 0, 0, 0, 0, 0, 0, 0, 1], [1, 0, 0, 0, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0....","{(0, 'BENIGN'): 1, (1, 'BENIGN'): 1, (2, 'BENI...",{'BENIGN'},"[{'id': 0, 'ip': '52.84.145.23'}, {'id': 1, 'i...",2017-07-07 17:00:00,2017-07-07 17:10:00,6
2015,2015,6,"[[0, 1], [2, 0], [3, 0], [3, 0], [4, 0], [5, 0]]","[[1, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 1, 0, 0, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0.6, 0.4, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0....","{(0, 'BENIGN'): 1, (1, 'BENIGN'): 1, (2, 'BENI...",{'BENIGN'},"[{'id': 0, 'ip': '192.168.10.8'}, {'id': 1, 'i...",2017-07-07 17:00:00,2017-07-07 17:10:00,6


In [51]:
graph_all_data_louvain['warn_num'].sum()

600519

In [52]:
multiple_risk_graph = graph_all_data_louvain[graph_all_data_louvain['group_label'].apply(lambda x: len(eval(x)) >= 2)]
multiple_risk_graph = multiple_risk_graph.reset_index(drop=True)
multiple_risk_graph_percent = len(multiple_risk_graph) / len(graph_all_data_louvain)
multiple_risk_alert_percent = multiple_risk_graph['warn_num'].sum() / graph_all_data_louvain['warn_num'].sum()

print('---------------------------------RESULTS----------------------------------------------------')
print("图划分单独的风险/攻击类型簇占比: {:.2%}, 告警占比{:.2%}".format(1-multiple_risk_graph_percent, 1-multiple_risk_alert_percent))

---------------------------------RESULTS----------------------------------------------------
图划分单独的风险/攻击类型簇占比: 98.17%, 告警占比49.49%


In [49]:
# 按照warn_num对multiple_risk_graph进行排序
multiple_risk_graph = multiple_risk_graph.sort_values(by='warn_num', ascending=False)
# 根据edge_label拿到每个label的数量
edges_label = multiple_risk_graph['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num_dict = {}
    for key,value in one_edge_label.items():
        label_name = key[1]
        if label_name in warn_num_dict.keys():
            warn_num_dict[label_name] += value
        else:
            warn_num_dict[label_name] = value
            
    warn_num_list.append(str(warn_num_dict))

multiple_risk_graph['warn_num_dicts'] = warn_num_list
multiple_risk_graph.reset_index(drop=True)
# id	group_num	edge_index	node_attr	edge_attr	group_attr	edge_label	group_label	nodes	start_time	end_time	warn_num	warn_num_dicts
multiple_risk_graph = multiple_risk_graph[['id','edge_index','group_num','node_attr','group_attr','edge_label','group_label','nodes','start_time','end_time','warn_num','warn_num_dicts']]

multiple_risk_graph.to_csv('multiple_attack_Leiden_graph.csv', index=False)

# our修正后，即考虑频繁项集结果

In [48]:
import pandas as pd
import numpy as np
import ast
# 训练集
coo_train_graph_df = pd.read_csv("../../datasets/null/our/train/graph_coo_data/coo_graph_10mins_correct.csv")
# 测试集
coo_test_graph_df = pd.read_csv("../../datasets/null/our/test/graph_coo_data/coo_graph_10mins_correct.csv")
# 按行拼接（垂直堆叠）
coo_graph_df = pd.concat([coo_train_graph_df, coo_test_graph_df], axis=0).reset_index(drop=True) # coo数据拼接

# 重新生成唯一id（覆盖原有id列）
coo_graph_df["id"] = range(len(coo_graph_df))

# 联合起来
graph_all_data_louvain = coo_graph_df
# 统计每个簇的告警数量
edges_label = graph_all_data_louvain['edge_label'].to_list()
warn_num_list = []
for one_edge in edges_label:
    one_edge_label = eval(one_edge)
    warn_num = sum(one_edge_label.values())
    warn_num_list.append(warn_num)
graph_all_data_louvain['warn_num'] = warn_num_list

# 去除unlabel影响,一个簇包含unlabel和另一个风险等级则认为该簇为这个风险等级，其他多风险等级的情况无法处理
for index_count in range(len(graph_all_data_louvain)):
    group_label = graph_all_data_louvain.loc[index_count,'group_label']
    group_label_set = eval(group_label)
    if 'unlabel' in group_label_set:
        group_label_set.remove('unlabel')
    if len(group_label_set) == 0:
        group_label_set = "{'unlabel'}"
    graph_all_data_louvain.loc[index_count,'group_label'] = str(group_label_set)

############################删除无标签数据##########################
graph_all_data_louvain = graph_all_data_louvain[graph_all_data_louvain['group_label'] != "{'unlabel'}"]
graph_all_data_louvain = graph_all_data_louvain.reset_index(drop=True)
graph_all_data_louvain

,id,group_num,edge_index,node_attr,edge_attr,group_attr,edge_label,group_label,Unnamed: 8,start_time,end_time,nodes,warn_num
0,0,0,"[[0, 1]]","[[1, 0, 1, 0], [0, 1, 1, 0]]","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","{(0, 'Benign'): 15}",{'Benign'},NaN,1999-03-29 08:00:00,1999-03-29 08:10:00,"[{'id': 0, 'ip': '192.168.1.10'}, {'id': 1, 'i...",15
1,1,1,"[[0, 1], [0, 2], [0, 3], [0, 4], [0, 5], [0, 6...","[[1, 0, 0, 1], [0, 1, 1, 0], [0, 1, 1, 0], [0,...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0....","{(0, 'Benign'): 2, (1, 'Benign'): 1, (2, 'Beni...",{'Benign'},NaN,1999-03-29 08:00:00,1999-03-29 08:10:00,"[{'id': 0, 'ip': '135.8.60.182'}, {'id': 1, 'i...",10
2,2,2,"[[0, 1], [0, 2], [0, 3]]","[[1, 0, 0, 1], [0, 1, 1, 0], [0, 1, 1, 0], [0,...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0....","{(0, 'Benign'): 1, (1, 'Benign'): 1, (2, 'Beni...",{'Benign'},NaN,1999-03-29 08:00:00,1999-03-29 08:10:00,"[{'id': 0, 'ip': '194.27.251.21'}, {'id': 1, '...",3
3,3,3,"[[0, 1]]","[[1, 0, 1, 0], [0, 1, 0, 1]]","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","{(0, 'Benign'): 1}",{'Benign'},NaN,1999-03-29 08:00:00,1999-03-29 08:10:00,"[{'id': 0, 'ip': '172.16.112.194'}, {'id': 1, ...",1
4,4,4,"[[0, 1], [0, 2], [0, 3], [0, 4], [0, 5], [0, 6...","[[1, 0, 0, 1], [0, 1, 1, 0], [0, 1, 1, 0], [0,...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0....","{(0, 'Benign'): 1, (1, 'Benign'): 1, (2, 'Beni...",{'Benign'},NaN,1999-03-29 08:00:00,1999-03-29 08:10:00,"[{'id': 0, 'ip': '195.115.218.108'}, {'id': 1,...",7
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7812,7812,4,"[[0, 1], [1, 0]]","[[1, 0, 1, 0], [0, 1, 1, 0]]","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","{(0, 'Benign'): 1, (1, 'Benign'): 1}",{'Benign'},NaN,1999-04-07 05:30:00,1999-04-07 05:40:00,"[{'id': 0, 'ip': '172.16.112.5'}, {'id': 1, 'i...",2
7813,7813,0,"[[0, 1], [0, 1], [0, 1], [0, 2]]","[[1, 0, 1, 0], [0, 1, 1, 0], [0, 1, 1, 0]]","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0....","{(0, 'Benign'): 1, (1, 'Benign'): 1, (2, 'Beni...",{'Benign'},NaN,1999-04-07 05:40:00,1999-04-07 05:50:00,"[{'id': 0, 'ip': '172.16.112.20'}, {'id': 1, '...",4
7814,7814,1,"[[0, 1], [1, 0]]","[[1, 0, 1, 0], [0, 1, 1, 0]]","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","{(0, 'Benign'): 1, (1, 'Benign'): 1}",{'Benign'},NaN,1999-04-07 05:40:00,1999-04-07 05:50:00,"[{'id': 0, 'ip': '172.16.112.5'}, {'id': 1, 'i...",2
7815,7815,0,"[[0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 2...","[[1, 0, 1, 0], [0, 1, 1, 0], [0, 1, 1, 0]]","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0....","{(0, 'Benign'): 1, (1, 'Benign'): 1, (2, 'Beni...",{'Benign'},NaN,1999-04-07 05:50:00,1999-04-07 06:00:00,"[{'id': 0, 'ip': '172.16.112.20'}, {'id': 1, '...",7


In [49]:
graph_all_data_louvain['warn_num'].sum()

251802

In [51]:
multiple_risk_graph = graph_all_data_louvain[graph_all_data_louvain['group_label'].apply(lambda x: len(eval(x)) >= 2)]
multiple_risk_graph = multiple_risk_graph.reset_index(drop=True)
multiple_risk_graph_percent = len(multiple_risk_graph) / len(graph_all_data_louvain)
multiple_risk_alert_percent = multiple_risk_graph['warn_num'].sum() / graph_all_data_louvain['warn_num'].sum()

print('---------------------------------RESULTS----------------------------------------------------')
print("图划分单独的风险/攻击类型簇占比: {:.2%}, 告警占比{:.2%}".format(1-multiple_risk_graph_percent, 1-multiple_risk_alert_percent))

---------------------------------RESULTS----------------------------------------------------
图划分单独的风险/攻击类型簇占比: 98.86%, 告警占比97.49%
